# 03. マッピングテーブル生成

IPC↔ECCN対応表・HS↔外為法対応表・規制キーワード辞書を生成する。

In [ ]:
DRY_RUN = False

import sys, os, logging
from pathlib import Path

try:
    BASE
except NameError:
    BASE        = Path("/content/AI_TradeManagement")
    STAGING_DIR = BASE / "data" / "staging"
    sys.path.insert(0, str(BASE / "scripts"))

try:
    ANTHROPIC_API_KEY
except NameError:
    try:
        from google.colab import userdata
        ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY") or ""
    except Exception:
        ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
CONTROL_NODES_PATH = BASE / "data" / "unified" / "control_nodes.json"
HS_CODES_PATH      = BASE / "data" / "hs2022_6digit.json"
print(f"DRY_RUN={DRY_RUN}, ANTHROPIC_API_KEY={'設定済み' if ANTHROPIC_API_KEY else '未設定'}")

## 1. IPC ↔ ECCN マッピング生成

In [ ]:
from pipeline.transform.mapping_builder import build_ipc_eccn_mapping

ipc_eccn_out = STAGING_DIR / "mappings" / "ipc_eccn_mapping.json"
ipc_eccn_map = build_ipc_eccn_mapping(
    control_nodes_path=CONTROL_NODES_PATH,
    anthropic_api_key=ANTHROPIC_API_KEY if ANTHROPIC_API_KEY else None,
    output_path=ipc_eccn_out,
    dry_run=DRY_RUN,
)
print(f"✅ IPC↔ECCN: {len(ipc_eccn_map)} IPCクラスをカバー")
for ipc, eccns in list(ipc_eccn_map.items())[:5]:
    print(f"  {ipc} → {eccns}")

## 2. HS ↔ 外為法 マッピング生成

In [ ]:
from pipeline.transform.mapping_builder import build_hs_fefta_mapping

hs_fefta_out = STAGING_DIR / "mappings" / "hs_fefta_mapping.json"
hs_fefta_map = build_hs_fefta_mapping(
    control_nodes_path=CONTROL_NODES_PATH,
    hs_codes_path=HS_CODES_PATH,
    output_path=hs_fefta_out,
    dry_run=DRY_RUN,
)
print(f"✅ HS↔外為法: {len(hs_fefta_map)} HSコードがマッチ")

## 3. 規制キーワード辞書生成

In [ ]:
from pipeline.transform.mapping_builder import build_regulatory_keyword_dict

kw_dict_out = STAGING_DIR / "mappings" / "regulatory_keyword_dict.json"
kw_dict = build_regulatory_keyword_dict(
    control_nodes_path=CONTROL_NODES_PATH,
    output_path=kw_dict_out,
    dry_run=DRY_RUN,
)
print(f"✅ 規制キーワード辞書: {len(kw_dict)} 用語")

## 4. サマリー

In [ ]:
print(f"IPC↔ECCN 対応表  : {len(ipc_eccn_map):>5} IPCクラス")
print(f"HS↔外為法 対応表  : {len(hs_fefta_map):>5} HSコード")
print(f"規制キーワード辞書 : {len(kw_dict):>5} 用語")
print("\n次のノートブック → 04_build_faiss_index.ipynb")